# Experiment Results Viewer
Laadt `Experiment_results.jsonl` en toont de **created_facts** per input en prompt.

In [1]:
# ── Installeer benodigde packages (eenmalig) ──────────────────────────────────
#%pip install pandas ipywidgets --quiet

In [7]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML
import ipywidgets as widgets

## 1. Bestand inladen

In [8]:
# ── Pad naar het JSONL-bestand ─────────────────────────────────────────────────
FILE_PATH = Path("Experiment_results.jsonl")   # pas aan indien nodig

records = []
with FILE_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"✅ {len(records)} records geladen uit '{FILE_PATH.name}'")

✅ 508 records geladen uit 'Experiment_results.jsonl'


## 2. Platte tabel bouwen

In [9]:
def extract_row(rec):
    """Haal de relevante velden uit één record."""
    prompt_examples = rec.get("prompt", [])
    # prompt is een lijst van strings; join ze
    prompt_str = " | ".join(str(p) for p in prompt_examples) if prompt_examples else ""

    # User-content uit het eerste query-blok
    user_content = ""
    queries = rec.get("queries", [])
    if queries and isinstance(queries[0], list):
        for msg in queries[0]:
            if msg.get("role") == "user":
                user_content = msg.get("content", "")
                break

    return {
        "model":         rec.get("model", ""),
        "input":         rec.get("input", ""),
        "prompt":        prompt_str,
        "user_content":  user_content,
        "created_facts": rec.get("created_facts", ""),
        "total_tokens":  rec.get("meta", {}).get("total_tokens", None),
    }

df = pd.DataFrame([extract_row(r) for r in records])
print(f"Kolommen : {list(df.columns)}")
print(f"Rijen    : {len(df)}")
df.head(3)

,model,input,prompt,user_content,created_facts,total_tokens
0,llama3.1:8b,"give a list of days that teams t1, t2, t3 and ...",",","[INPUT]give a list of days that teams t1, t2, ...",playingday_request(t1).\nplayingday_request(t2...,404
1,llama3.1:8b,"give a list of days that teams t1, t2, t3 and ...",[INPUT]Add team t2 to the mens league[/INPUT] ...,"[INPUT]give a list of days that teams t1, t2, ...","schedule_request([t1,t2,t3,t4]).",357
2,llama3.1:8b,"give a list of days that teams t1, t2, t3 and ...",[INPUT]Add playing day 12[/INPUT] becomes [OUT...,"[INPUT]give a list of days that teams t1, t2, ...","schedule_request([t1, t2, t3, t4]).",357


## 3. Overzichtstabel — input × prompt × created_facts

In [10]:
CSS = """
<style>
.exp-table { border-collapse: collapse; width: 100%; font-family: sans-serif; font-size: 13px; }
.exp-table th { background:#4472C4; color:white; text-align:left; padding:6px 10px; }
.exp-table td { vertical-align:top; padding:5px 10px; border:1px solid #ccc;
                white-space:pre-wrap; max-width:420px; word-break:break-word; }
.exp-table tr:nth-child(even) td { background:#EEF2FA; }
</style>
"""

def style_table(df_sub):
    """Geeft een gestylede HTML-tabel terug (geen jinja2 nodig)."""
    cols = ["model", "input", "prompt", "created_facts", "total_tokens"]
    rows = ""
    for _, row in df_sub[cols].iterrows():
        cells = "".join(f"<td>{str(row[c])}</td>" for c in cols)
        rows += f"<tr>{cells}</tr>"
    headers = "".join(f"<th>{c}</th>" for c in cols)
    html = f"{CSS}<table class='exp-table'><thead><tr>{headers}</tr></thead><tbody>{rows}</tbody></table>"
    return HTML(html)

#display(style_table(df))

## 4. Interactief filteren op input-tekst

In [5]:
search_box = widgets.Text(
    placeholder="Zoek op input-tekst…",
    description="Filter:",
    layout=widgets.Layout(width="60%")
)
out = widgets.Output()

def on_change(change):
    out.clear_output(wait=True)
    term = change["new"].strip().lower()
    mask = df["input"].str.lower().str.contains(term, na=False) if term else pd.Series([True]*len(df))
    with out:
        display(style_table(df[mask]))

search_box.observe(on_change, names="value")

with out:
    display(style_table(df))

display(search_box, out)

Output()

## 5. Detail per record — dropdown

In [11]:
options = [(f"[{i}] {row['input'][:70]}", i) for i, row in df.iterrows()]

dropdown = widgets.Dropdown(
    options=options,
    description="Record:",
    layout=widgets.Layout(width="80%")
)
detail_out = widgets.Output()

def show_detail(change):
    idx = change["new"]
    row = df.iloc[idx]
    rec = records[idx]
    detail_out.clear_output(wait=True)
    with detail_out:
        display(HTML(f"""
        <div style='font-family:monospace; border:1px solid #4472C4;
                    border-radius:6px; padding:14px; background:#f7f9ff'>
          <h3 style='color:#4472C4'>Record {idx}</h3>
          <b>Model:</b> {row['model']}<br>
          <b>Total tokens:</b> {row['total_tokens']}<br><br>

          <b>Input:</b><br>
          <pre style='background:#fff;padding:8px;border:1px solid #ddd'>{row['input']}</pre>

          <b>Prompt (few-shot voorbeelden):</b><br>
          <pre style='background:#fff;padding:8px;border:1px solid #ddd'>{row['prompt'] or '(leeg)'}</pre>

          <b>Created facts:</b><br>
          <pre style='background:#e8f5e9;padding:8px;border:1px solid #a5d6a7;
                      color:#1b5e20;font-size:13px'>{row['created_facts']}</pre>
        </div>
        """))

dropdown.observe(show_detail, names="value")
show_detail({"new": 0})   # toon eerste record meteen

display(dropdown, detail_out)

Output()

## 6. Statistieken

In [12]:
print("=== Algemene statistieken ===")
print(f"Totaal records       : {len(df)}")
print(f"Unieke inputs        : {df['input'].nunique()}")
print(f"Unieke modellen      : {df['model'].nunique()} → {df['model'].unique()}")
print(f"Gem. tokens/record   : {df['total_tokens'].mean():.1f}")
print()

print("=== Records per model ===")
display(df.groupby("model").agg(
    aantal=("input", "count"),
    gem_tokens=("total_tokens", "mean")
).round(1))

print()
print("=== Top-5 meest voorkomende inputs ===")
display(df["input"].value_counts().head(5))

input
give a list of days that teams t1, t2, t3 and t4 play.          127
Is team t2 playing on day 3 and is team t3 playing on day 4?    127
When is team 't1' playing?                                      127
Give me the full schedule                                       127
Name: count, dtype: int64

## 7. Export gefilterde resultaten naar CSV

In [13]:
export_path = Path("experiment_results_export.csv")
df[["model", "input", "prompt", "created_facts", "total_tokens"]].to_csv(
    export_path, index=False, encoding="utf-8-sig"
)
print(f"✅ Geëxporteerd naar '{export_path}'")

✅ Geëxporteerd naar 'experiment_results_export.csv'
